# 106 — Audit SmolVLA Q10 corrector training

Evaluate saved checkpoints on the same held-out v3 tree roots. These roots were split by tree identity before training. Report stock, oracle, within-tree ordering, rescues, spoils, and fixed argmax selection. The old 95 arms use demonstrator pretraining plus 50% demonstrator TD replay at gamma=.99; the new 107 arms use tree-only terminal-success targets at gamma=1. The scores answer different value questions, so compare selected actions and outcomes rather than their absolute Q values.

Any threshold chosen after looking at these outcomes is exploratory. A deployment threshold needs a separate tuning split and fresh-identity audit.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')
EXTRAS = 'sim'
SETUP_ENV = True
import urllib.request
exec(urllib.request.urlopen(
    'https://raw.githubusercontent.com/ArjunS07/cs159-sp26/main/pnp-vla/scripts/colab_bootstrap.py'
).read().decode())


In [ ]:
from pathlib import Path
import glob
import numpy as np
import pandas as pd
import torch
from pnp.store import SupabaseStore
from pnp.qplanning_critic.config import QPlanningModelConfig
from pnp.qplanning_critic.model import QPlanningCritic
from pnp.smolvla_tree_bellman_finetune import (
    DEFAULT_SNAPSHOT_KEY, RootTreeDataset, _evaluate_roots,
    _model_from_checkpoint, load_or_create_tree_snapshot,
    prepare_tree_bellman_cache)
from pnp.smolvla_success_critic import FORMAT, TimedRoots, evaluate_roots

TREE_LIMIT = 480
OLD_ROOT = Path('/content/drive/MyDrive/pnp_smolvla_tree_bellman_finetune')
NEW_ROOT = Path('/content/drive/MyDrive/pnp_smolvla_success_q10')
CACHE_ROOT = Path('/content/smolvla_success_tree_cache')
store = SupabaseStore()
snapshot = load_or_create_tree_snapshot(
    store=store, snapshot_key=DEFAULT_SNAPSHOT_KEY, tree_limit=TREE_LIMIT)
cache = prepare_tree_bellman_cache(
    snapshot=snapshot, cache_root=CACHE_ROOT, gamma=1.0,
    success_reward=True, store=store)
groups = {item['candidate_group_id']: item for item in snapshot['groups']}
validation = cache['validation_group_ids']
old_roots = RootTreeDataset(cache, validation)
new_roots = TimedRoots(cache, validation, groups)
assert set(cache['train_group_ids']).isdisjoint(validation)
assert len(old_roots) == len(new_roots) == len(validation)
stock = np.asarray([old_roots[i]['success'][0] for i in range(len(old_roots))], bool)
oracle = np.asarray([old_roots[i]['success'].any() for i in range(len(old_roots))], bool)
print({'snapshot': snapshot['snapshot_digest'], 'trees': len(snapshot['groups']),
       'validation_trees': len(validation), 'stock_successes': int(stock.sum()),
       'oracle_successes': int(oracle.sum()),
       'recoverable_stock_failures': int((~stock & oracle).sum())})


In [ ]:
def unique_checkpoint(root, arm):
    matches = sorted(glob.glob(str(root / snapshot['snapshot_digest'] / arm
                                   / 'checkpoint_step_*.pt')))
    return Path(matches[-1]) if matches else None

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
rows = []
for arm in ('bellman_control', 'fork_aware', 'root_mc', 'tree_td'):
    old = arm in ('bellman_control', 'fork_aware')
    path = unique_checkpoint(OLD_ROOT if old else NEW_ROOT, arm)
    if path is None:
        print(f'{arm}: no checkpoint found; skipped')
        continue
    payload = torch.load(path, map_location='cpu', weights_only=False)
    assert payload['snapshot_digest'] == snapshot['snapshot_digest']
    if old:
        assert payload['format'] == 'smolvla_tree_bellman_finetune_v1'
        model, _ = _model_from_checkpoint(payload, device)
        metrics = _evaluate_roots(model, old_roots, device)
    else:
        assert payload['format'] == FORMAT
        assert payload['gamma'] == 1.0 and payload['reward'] == 'step_success'
        architecture = dict(payload['architecture'])
        dims = {key: architecture.pop(key)
                for key in ('prefix_dim', 'robot_dim', 'proprio_dim')}
        model = QPlanningCritic(config=QPlanningModelConfig(**architecture),
                                **dims).to(device)
        model.load_state_dict(payload['model'])
        metrics = evaluate_roots(model, new_roots, device)
    rows.append({'arm': arm, 'update': payload['update'], **metrics})
    del model
report = pd.DataFrame(rows)
display(report)


## Interpretation

A higher held-out selection rate is evidence about **one intervention followed by the frozen continuation policy** on this sampled root distribution. Each candidate has one recorded continuation seed. Use a fresh branch audit before claiming a reliable action effect, and a full closed-loop rollout before claiming policy improvement.
